# 02 — Préparation des données

**Objectif :** préparer la matrice client-produit sur l’ensemble des identifiants. Le segment initial et la conservation de la première occurrence restent des références de comparaison.

## 1. Import des fonctions et chargement

Le CSV original doit être placé dans `data/données.csv`.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Fonctionne que Jupyter démarre dans la racine ou dans notebooks/
racine = Path.cwd()
if not (racine / 'src').is_dir():
    racine = racine.parent
if not (racine / 'src').is_dir():
    raise FileNotFoundError('Ouvrir le notebook depuis le projet ou notebooks/.')
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.chargement import charger_donnees, colonnes_produits
from src.preparation import (
    verifier_produits, decouper_a_la_premiere_rupture,
    base_initiale, base_agregee,
    frequences_produits, comparer_frequences,
    base_identifiants_distincts, diagnostic_identifiants, verifier_base_agregee,
)

df = charger_donnees(racine)
produits = colonnes_produits(df)
verifier_produits(df, produits)
print(f'Données : {df.shape[0]} lignes, {len(produits)} produits')

Données : 73930 lignes, 26 produits


## 2. Définition des populations et contrôle des identifiants

**Hypothèse principale :** pour un même identifiant, un produit est considéré présent si au moins une ligne indique 1 (`max`). Cette règle de consolidation reste une hypothèse de travail.

**Comparaisons :** segment initial et première occurrence de chaque identifiant. Deux identifiants différents ne sont jamais fusionnés, même si leurs profils sont identiques.

In [3]:
# Diagnostic des identifiants avant transformation
display(diagnostic_identifiants(df, produits))

# Construction des trois matrices
segment_initial, segment_final, limite = decouper_a_la_premiere_rupture(df)
matrice_initiale = base_initiale(df, produits)
matrice_distincte = base_identifiants_distincts(df, produits)
matrice_agregee = base_agregee(df, produits)

# Contrôles techniques : identifiants conservés, unicité et produits binaires
verifier_base_agregee(df, produits, matrice_agregee)

bilan = pd.DataFrame({
    'scenario': ['Données complètes agrégées (principale)',
                 'Segment initial (comparaison)',
                 'Première occurrence (comparaison)'],
    'nombre_clients': [len(matrice_agregee), len(matrice_initiale), len(matrice_distincte)],
    'nombre_produits': [matrice_agregee.shape[1], matrice_initiale.shape[1], matrice_distincte.shape[1]],
})
display(bilan)
display(matrice_agregee.head())

# Exemple de contrôle ciblé : identifiant dont les profils diffèrent
print('Profil agrégé de l’ID 72521 :')
display(matrice_agregee.loc[[72521]])

,indicateur,valeur
0,observations_initiales,73930
1,identifiants_distincts,72638
2,identifiants_repetes,118
3,lignes_excedentaires_par_id,1292
4,ids_repetes_profils_produits_differents,12


,scenario,nombre_clients,nombre_produits
0,Données complètes agrégées (principale),72638,26
1,Segment initial (comparaison),72531,26
2,Première occurrence (comparaison),72638,26


,CART,CJEU,PACKSERV,DAV,DECAUTH,CPAI,CARTE,auto,GAV,PJ,...,ASSMH,CREDHAB,CSL,OBLIGATION,SICAVFCP,EPARBANC,ASSPREVDEC,ASSDEP,CREDRENOV,ASSPREVDECDEP
id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,1,0,0,0,1,0,0,...,1,0,1,0,0,1,0,0,0,0
2,1,0,1,1,0,1,0,1,1,1,...,1,0,1,0,1,1,0,0,0,0
3,1,1,1,1,1,1,0,0,1,1,...,1,0,1,0,0,0,0,0,0,0
4,1,0,1,1,1,1,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0
5,1,0,0,1,0,1,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0


Profil agrégé de l’ID 72521 :


,CART,CJEU,PACKSERV,DAV,DECAUTH,CPAI,CARTE,auto,GAV,PJ,...,ASSMH,CREDHAB,CSL,OBLIGATION,SICAVFCP,EPARBANC,ASSPREVDEC,ASSDEP,CREDRENOV,ASSPREVDECDEP
id,,,,,,,,,,,,,,,,,,,,,
72521,1,0,1,1,1,1,1,1,1,1,...,1,0,1,1,0,1,0,0,0,0


## 3. Fréquence de détention des produits

Chaque pourcentage correspond à la part des clients de la population retenue qui détiennent le produit.

In [4]:
display(frequences_produits(matrice_agregee).round(2))

,pourcentage_clients
DAV,99.04
CART,85.01
CSL,84.28
PACKSERV,82.80
CPAI,79.85
ASSMH,64.29
DECAUTH,64.16
CREDCONSO,44.23
PJ,42.54
EPARLOG,36.89


## 4. Comparaison descriptive des scénarios

Les pourcentages sont calculés pour chacune des trois populations. Les écarts sont exprimés en points de pourcentage par rapport à la base complète agrégée.

In [5]:
frequences = pd.DataFrame({
    'Données complètes agrégées': matrice_agregee.mean() * 100,
    'Segment initial': matrice_initiale.mean() * 100,
    'Première occurrence': matrice_distincte.mean() * 100,
})
frequences['Écart initial vs agrégée (pp)'] = (
    frequences['Segment initial'] - frequences['Données complètes agrégées']
)
frequences['Écart première occurrence vs agrégée (pp)'] = (
    frequences['Première occurrence'] - frequences['Données complètes agrégées']
)
display(frequences.round(3))

,Données complètes agrégées,Segment initial,Première occurrence,Écart initial vs agrégée (pp),Écart première occurrence vs agrégée (pp)
CART,85.008,84.982,85.004,-0.026,-0.004
CJEU,11.051,11.067,11.051,0.016,0.000
PACKSERV,82.798,82.766,82.791,-0.032,-0.007
DAV,99.042,99.039,99.040,-0.003,-0.001
DECAUTH,64.155,64.087,64.140,-0.068,-0.015
CPAI,79.852,79.813,79.843,-0.039,-0.010
CARTE,1.753,1.591,1.736,-0.161,-0.017
auto,33.054,32.943,33.042,-0.111,-0.012
GAV,10.614,10.466,10.598,-0.148,-0.017
PJ,42.536,42.437,42.522,-0.099,-0.014


## 5. Points à interpréter après exécution

- Vérifier que la matrice principale contient autant de lignes que d’identifiants distincts.
- Examiner l’influence de l’agrégation sur les produits dont les profils sont contradictoires pour un même ID.
- Comparer les fréquences entre scénarios avant de recalculer les associations.
- Documenter la règle d’agrégation choisie et ses limites dans le dossier.

**Étape suivante :** adapter le notebook 03 à la population principale retenue, puis recalculer support, confiance, confiance attendue et lift.